In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
# This creates a folder named 'Panoptic_Checkpoints' right in your main Google Drive
checkpoint_dir = '/content/drive/MyDrive/Panoptic_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Weights will be permanently saved to: {checkpoint_dir}")

Weights will be permanently saved to: /content/drive/MyDrive/Panoptic_Checkpoints


In [ ]:
from google.colab import files
import os

# 1. Upload your kaggle.json key
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# 2. Move the key to the hidden Kaggle directory
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API successfully authenticated!")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API successfully authenticated!


In [ ]:
# 1. Download the dataset using the correct underscore slug
!kaggle datasets download -d solesensei/solesensei_bdd100k

# 2. Unzip it directly into Colab's high-speed local disk
!unzip -q solesensei_bdd100k.zip -d /content/DATASETS

print("Dataset downloaded and extracted!")

Dataset URL: https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k
License(s): other
100% 7.61G/7.61G [06:40<00:00, 20.4MB/s]

Dataset downloaded and extracted!


In [ ]:
import torch
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, in_channels=256, num_classes=1):
        super().__init__()

        self.relu = nn.ReLU(inplace=True)

        self.upconv1 = nn.ConvTranspose2d(in_channels, 128, kernel_size=2, stride=2)
        self.conv1 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(num_features=128, dtype=torch.float32)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(num_features=64, dtype=torch.float32)

        self.upconv3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(num_features=32, dtype=torch.float32)

        self.finalconv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, P2):
        x = self.upconv1(P2)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x) 

        x = self.upconv2(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        x = self.upconv3(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)

        x = self.finalconv(x)

        return x

In [ ]:
import math
import torch
import torch.nn as nn

class DecoupledYOLO(nn.Module):
    def __init__(self, in_channels=256, num_anchors=3, num_classes=10):
        super().__init__()
        self.num_anchors = num_anchors
        self.num_classes = num_classes

        self.heads = nn.ModuleList()
        for _ in range(3):  # For N2, N3, N4 scales
            stem = nn.Sequential(
                nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(in_channels),
                nn.SiLU(inplace=True)
            )
            reg_branch = nn.Conv2d(in_channels, num_anchors * 5, kernel_size=1)
            cls_branch = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=1)


            prior_prob = 0.01
            bias_value = math.log(prior_prob / (1.0 - prior_prob))

            
            nn.init.constant_(reg_branch.bias.data, 0.0)
            for a in range(num_anchors):

                reg_branch.bias.data[a * 5 + 4] = bias_value
                nn.init.constant_(cls_branch.bias.data, bias_value)

            self.heads.append(nn.ModuleDict({'stem': stem, 'reg': reg_branch, 'cls': cls_branch}))

    def forward(self, N2, N3, N4):
        features = [N2, N3, N4]
        outputs = []

        for i, feat in enumerate(features):
            x = self.heads[i]['stem'](feat)
            reg_out = self.heads[i]['reg'](x)
            cls_out = self.heads[i]['cls'](x)

            B, _, H, W = feat.shape
            reg_out = reg_out.view(B, self.num_anchors, 5, H, W)
            cls_out = cls_out.view(B, self.num_anchors, self.num_classes, H, W)

            
            combined = torch.cat([reg_out, cls_out], dim=2)
            combined = combined.contiguous().view(B, self.num_anchors * (5 + self.num_classes), H, W)
            outputs.append(combined)

        return outputs[0], outputs[1], outputs[2]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights

class PanopticPerceptionEngine(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet34(weights=ResNet34_Weights.DEFAULT)

        self.layer2 = nn.Sequential(*list(resnet.children())[0:6])
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        out_channels = 256

        self.latlayer2 = nn.Conv2d(128, out_channels, kernel_size=1)
        self.latlayer3 = nn.Conv2d(256, out_channels, kernel_size=1)
        self.latlayer4 = nn.Conv2d(512, out_channels, kernel_size=1)

        self.smooth2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.downsample_p2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True)
        )
        self.downsample_p3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True)
        )

        self.unet_head = UNet(in_channels=out_channels, num_classes=1)
        self.yolo_head = DecoupledYOLO(in_channels=out_channels, num_anchors=3, num_classes=10)

    def _upsample_add(self, x, y):
        _, _, H, W = y.size()
        return F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False) + y

    def forward(self, x):
        c2 = self.layer2(x)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)

        P4 = self.latlayer4(c4)
        p3_proj = self.latlayer3(c3)
        P3 = self.smooth3(self._upsample_add(P4, p3_proj))
        p2_proj = self.latlayer2(c2)
        P2 = self.smooth2(self._upsample_add(P3, p2_proj))

        N2 = P2
        N3 = P3 + self.downsample_p2(N2)
        N4 = P4 + self.downsample_p3(N3)

        seg_mask = self.unet_head(P2)

        yolo_box = self.yolo_head(N2, N3, N4)

        return seg_mask, yolo_box

In [ ]:
import numpy as np
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment
import torch

class Collision:
    def __init__(self, max_distance=50, max_disappeared=5):
        self.next_object_id = 0
        self.objects = {}       
        self.disappeared = {} 
        self.max_distance = max_distance
        self.max_disappeared = max_disappeared

    def register(self, centroid):
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def deregister(self, object_id):
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update_tracker(self, bounding_boxes):
        if len(bounding_boxes) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects

        input_centroids = []
        for box in bounding_boxes:
            cx = int((box[0] + box[2]) / 2.0)
            cy = int((box[1] + box[3]) / 2.0)
            input_centroids.append((cx, cy))

        if len(self.objects) == 0:
            for centroid in input_centroids:
                self.register(centroid)
            return self.objects

        object_ids = list(self.objects.keys())
        object_centroids = list(self.objects.values())

        D = distance.cdist(np.array(object_centroids), np.array(input_centroids))
        row_indices, col_indices = linear_sum_assignment(D)

        used_rows = set()
        used_cols = set()

        for row, col in zip(row_indices, col_indices):
            if D[row, col] > self.max_distance:
                continue

            object_id = object_ids[row]
            self.objects[object_id] = input_centroids[col]
            self.disappeared[object_id] = 0

            used_rows.add(row)
            used_cols.add(col)

        unused_rows = set(range(D.shape[0])).difference(used_rows)
        for row in unused_rows:
            object_id = object_ids[row]
            self.disappeared[object_id] += 1
            if self.disappeared[object_id] > self.max_disappeared:
                self.deregister(object_id)

        unused_cols = set(range(len(input_centroids))).difference(used_cols)
        for col in unused_cols:
            self.register(input_centroids[col])

        return self.objects

    def check_collision(self, detections, segmentation_mask):
        warnings = []

        if isinstance(segmentation_mask, torch.Tensor):
            segmentation_mask = segmentation_mask.detach().cpu()

        _, _, max_h, max_w = segmentation_mask.shape

        ignored_classes = [8, 9]

        for det in detections:
            if len(det) >= 6:
                x1, y1, x2, y2, score, cls = det
                if int(cls) in ignored_classes:
                    continue
            else:
                x1, y1, x2, y2 = det[:4]

            cx = int((x1 + x2) / 2.0)
            cy_bottom = int(y2)

            if 0 <= cx < max_w and 0 <= cy_bottom < max_h:
                if isinstance(segmentation_mask, torch.Tensor):
                    pixel_value = segmentation_mask[0, 0, cy_bottom, cx].item()
                else:
                    pixel_value = segmentation_mask[0, 0, cy_bottom, cx]

                if pixel_value == 1:
                    warnings.append(det)

        return warnings

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        p = torch.sigmoid(inputs)
        pt = p * targets + (1.0 - p) * (1.0 - targets)
        pt = torch.clamp(pt, min=1e-6, max=1.0 - 1e-6)
        alpha_factor = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        return (alpha_factor * ((1.0 - pt) ** self.gamma) * bce_loss).mean()

class PanopticLoss(nn.Module):
    def __init__(self, yolo_weight=2.0, unet_weight=1.0, num_classes=10):
        super().__init__()
        self.yolo_weight = yolo_weight
        self.unet_weight = unet_weight
        self.num_classes = num_classes

        self.unet_focal = FocalLoss(alpha=0.25, gamma=2.0)
        self.yolo_class_focal = FocalLoss(alpha=0.25, gamma=2.0)
        self.yolo_obj_bce = nn.BCEWithLogitsLoss()

    def dice_loss(self, preds, targets, smooth=1e-6):
        preds = torch.sigmoid(preds)
        intersection = (preds * targets).sum(dim=(2, 3))
        union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        return 1.0 - ((2.0 * intersection + smooth) / (union + smooth)).mean()

    def compute_ciou(self, pred_xy, pred_wh, target_xy, target_wh):
        pred_wh = torch.clamp(pred_wh, max=20.0, min=-20.0)

        p_w, p_h = torch.exp(pred_wh[:, 0]), torch.exp(pred_wh[:, 1])
        t_w, t_h = torch.exp(target_wh[:, 0]), torch.exp(target_wh[:, 1])

        p_x1, p_x2 = pred_xy[:, 0] - p_w / 2, pred_xy[:, 0] + p_w / 2
        p_y1, p_y2 = pred_xy[:, 1] - p_h / 2, pred_xy[:, 1] + p_h / 2

        t_x1, t_x2 = target_xy[:, 0] - t_w / 2, target_xy[:, 0] + t_w / 2
        t_y1, t_y2 = target_xy[:, 1] - t_h / 2, target_xy[:, 1] + t_h / 2

        inter_area = torch.clamp(torch.min(p_x2, t_x2) - torch.max(p_x1, t_x1), min=0) * \
                     torch.clamp(torch.min(p_y2, t_y2) - torch.max(p_y1, t_y1), min=0)

        iou = inter_area / ((p_w * p_h) + (t_w * t_h) - inter_area + 1e-6)

        c2 = torch.clamp(torch.max(p_x2, t_x2) - torch.min(p_x1, t_x1), min=0) ** 2 + \
             torch.clamp(torch.max(p_y2, t_y2) - torch.min(p_y1, t_y1), min=0) ** 2 + 1e-6

        rho2 = (pred_xy[:, 0] - target_xy[:, 0]) ** 2 + (pred_xy[:, 1] - target_xy[:, 1]) ** 2
        v = (4 / (math.pi ** 2)) * torch.pow(torch.atan(t_w / (t_h + 1e-6)) - torch.atan(p_w / (p_h + 1e-6)), 2)

        with torch.no_grad():
            alpha = v / (1.0 - iou + v + 1e-6)

        return (1.0 - (iou - (rho2 / c2 + v * alpha))).mean()

    def forward(self, unet_preds, unet_targets, has_masks, yolo_preds, yolo_targets):
        
        if has_masks.any():
            valid_preds = unet_preds[has_masks]
            valid_targets = unet_targets[has_masks]

            
            if valid_preds.shape[-2:] != valid_targets.shape[-2:]:
                valid_preds = F.interpolate(
                    valid_preds,
                    size=valid_targets.shape[-2:],
                    mode='bilinear',
                    align_corners=False
                )

            loss_unet = self.unet_focal(valid_preds, valid_targets) + \
                        self.dice_loss(valid_preds, valid_targets)
        else:
            loss_unet = unet_preds.sum() * 0.0
        

        loss_yolo = 0.0
        for pred, target in zip(yolo_preds, yolo_targets):
            B, C, H, W = pred.shape

            pred = pred.contiguous().view(B, 3, 5 + self.num_classes, H, W)
            target = target.contiguous().view(B, 3, 5 + self.num_classes, H, W)

            obj_mask = target[:, :, 4, :, :] == 1.0
            loss_yolo += self.yolo_obj_bce(pred[:, :, 4, :, :], target[:, :, 4, :, :])

            if obj_mask.sum() > 0:
                pred_obj = pred.permute(0, 1, 3, 4, 2)[obj_mask]
                target_obj = target.permute(0, 1, 3, 4, 2)[obj_mask]

                loss_yolo += 5.0 * self.compute_ciou(
                    torch.sigmoid(pred_obj[:, :2]), pred_obj[:, 2:4],
                    target_obj[:, :2], target_obj[:, 2:4]
                )
                loss_yolo += self.yolo_class_focal(pred_obj[:, 5:], target_obj[:, 5:])

        return (self.yolo_weight * loss_yolo) + (self.unet_weight * loss_unet), loss_unet, loss_yolo

In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

IMG_H, IMG_W = 384, 640
NUM_CLASSES = 10
NUM_ANCHORS = 3
STRIDES = [8, 16, 32]
GRID_SIZES = [(IMG_H // s, IMG_W // s) for s in STRIDES]

BDD_CLASSES = ['person', 'rider', 'car', 'truck', 'bus', 'train', 'motor', 'bike', 'traffic light', 'traffic sign']
CLASS_TO_IDX = {c: i for i, c in enumerate(BDD_CLASSES)}

ANCHORS = [
    [(10, 13), (16, 30), (33, 23)],
    [(30, 61), (62, 45), (59, 119)],
    [(116, 90), (156, 198), (373, 326)]
]


DATASET_ROOT = '/content/DATASETS'

IMAGES_TRAIN = f'{DATASET_ROOT}/bdd100k/bdd100k/images/100k/train'
IMAGES_VAL   = f'{DATASET_ROOT}/bdd100k/bdd100k/images/100k/val'

DET_JSON_TRAIN = f'{DATASET_ROOT}/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json'
DET_JSON_VAL   = f'{DATASET_ROOT}/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json'

DRIVABLE_MASK_TRAIN = f'{DATASET_ROOT}/bdd100k_seg/bdd100k/seg/labels/train'
DRIVABLE_MASK_VAL   = f'{DATASET_ROOT}/bdd100k_seg/bdd100k/seg/labels/val'

class BDD100KDataset(Dataset):
    def __init__(self, images_dir, det_json_path, drivable_mask_dir, split='train', transform=None):
        self.images_dir = images_dir
        self.drivable_mask_dir = drivable_mask_dir
        self.split = split
        self.transform = transform or T.Compose([
            T.Resize((IMG_H, IMG_W)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        
        self.image_path_map = {}
        for root, _, files in os.walk(self.images_dir):
            for file in files:
                if file.endswith(('.jpg', '.png')):
                    self.image_path_map[file] = os.path.join(root, file)

        with open(det_json_path, 'r') as f:
            self.annotations = json.load(f)

        self.annotations = [
            a for a in self.annotations
            if a['name'] in self.image_path_map
        ]
        print(f"[{split.upper()}] Successfully mapped {len(self.annotations)} valid images.")

    def __len__(self):
        return len(self.annotations)

    def _load_boxes(self, entry):
        boxes = []
        for label in entry.get('labels', []):
            cat = label.get('category')
            if cat not in CLASS_TO_IDX or 'box2d' not in label:
                continue
            b = label['box2d']
            boxes.append((b['x1'], b['y1'], b['x2'], b['y2'], CLASS_TO_IDX[cat]))
        return boxes

    def _build_targets(self, boxes, orig_w, orig_h):
        targets = [torch.zeros(NUM_ANCHORS * (5 + NUM_CLASSES), gh, gw)
                   for (gh, gw) in GRID_SIZES]

        sx, sy = IMG_W / orig_w, IMG_H / orig_h

        for (x1, y1, x2, y2, cls) in boxes:
            x1, x2 = x1 * sx, x2 * sx
            y1, y2 = y1 * sy, y2 * sy
            bw, bh = x2 - x1, y2 - y1
            cx, cy = x1 + bw / 2, y1 + bh / 2
            if bw <= 1 or bh <= 1:
                continue

            best_scale, best_anchor, best_iou = 0, 0, -1
            for s_idx, anchors in enumerate(ANCHORS):
                for a_idx, (aw, ah) in enumerate(anchors):
                    inter = min(bw, aw) * min(bh, ah)
                    union = bw * bh + aw * ah - inter
                    iou = inter / union
                    if iou > best_iou:
                        best_iou, best_scale, best_anchor = iou, s_idx, a_idx

            gh, gw = GRID_SIZES[best_scale]
            stride = STRIDES[best_scale]
            gi, gj = int(cx / stride), int(cy / stride)
            gi, gj = min(gi, gw - 1), min(gj, gh - 1)

            aw, ah = ANCHORS[best_scale][best_anchor]
            tx = (cx / stride) - gi
            ty = (cy / stride) - gj
            tw = np.log(max(bw, 1e-6) / aw)
            th = np.log(max(bh, 1e-6) / ah)

            ch0 = best_anchor * (5 + NUM_CLASSES)
            t = targets[best_scale]
            t[ch0 + 0, gj, gi] = tx
            t[ch0 + 1, gj, gi] = ty
            t[ch0 + 2, gj, gi] = tw
            t[ch0 + 3, gj, gi] = th
            t[ch0 + 4, gj, gi] = 1.0
            t[ch0 + 5 + cls, gj, gi] = 1.0

        return targets

    def __getitem__(self, idx):
        entry = self.annotations[idx]
        img_path = self.image_path_map[entry['name']]
        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size

        boxes = self._load_boxes(entry)
        box_targets = self._build_targets(boxes, orig_w, orig_h)

        image = self.transform(image)

        base_name = os.path.splitext(entry['name'])[0]

        
        possible_mask_filenames = [
            f"{base_name}_{self.split}_id.png",
            f"{base_name}_drivable.png",
            f"{base_name}.png"
        ]

        mask_path = None
        for m_name in possible_mask_filenames:
            candidate_path = os.path.join(self.drivable_mask_dir, m_name)
            if os.path.exists(candidate_path):
                mask_path = candidate_path
                break

        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path)
            mask = mask.resize((IMG_W, IMG_H), Image.Resampling.NEAREST)
            mask = np.array(mask)
            mask = ((mask == 1) | (mask == 2)).astype(np.float32)
            has_mask = True
        else:
            mask = np.zeros((IMG_H, IMG_W), dtype=np.float32)
            has_mask = False

        mask_target = torch.from_numpy(mask).unsqueeze(0)
        return image, mask_target, torch.tensor(has_mask, dtype=torch.bool), box_targets


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    has_masks = torch.stack([b[2] for b in batch])
    box_targets = [
        torch.stack([b[3][scale_idx] for b in batch])
        for scale_idx in range(len(GRID_SIZES))
    ]
    return images, masks, has_masks, box_targets

robust_train_transform = T.Compose([
    T.Resize((IMG_H, IMG_W)),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = BDD100KDataset(
    images_dir=IMAGES_TRAIN,
    det_json_path=DET_JSON_TRAIN,
    drivable_mask_dir=DRIVABLE_MASK_TRAIN,
    split='train',
    transform=robust_train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_fn
)

[TRAIN] Successfully mapped 69863 valid images.


In [ ]:
import torch
from torchvision.ops import batched_nms

def decode_yolo_predictions(yolo_preds, conf_threshold=0.3, nms_threshold=0.5, max_detections=300):
    device = yolo_preds[0].device
    B = yolo_preds[0].shape[0]
    batch_detections = [[] for _ in range(B)]

    for scale_idx, pred in enumerate(yolo_preds):
        stride = STRIDES[scale_idx]
        anchors = torch.tensor(ANCHORS[scale_idx], device=device, dtype=torch.float32)
        Bc, C, H, W = pred.shape

        pred = pred.contiguous().view(Bc, NUM_ANCHORS, 5 + NUM_CLASSES, H, W).permute(0, 1, 3, 4, 2)

        tx = pred[..., 0]
        ty = pred[..., 1]
        tw = pred[..., 2]
        th = pred[..., 3]
        obj = torch.sigmoid(pred[..., 4])
        cls_scores = torch.sigmoid(pred[..., 5:])

        gi = torch.arange(W, device=device).view(1, 1, 1, W).float()
        gj = torch.arange(H, device=device).view(1, 1, H, 1).float()

        bx = (torch.sigmoid(tx) + gi) * stride
        by = (torch.sigmoid(ty) + gj) * stride

        tw = torch.clamp(tw, min=-20.0, max=20.0)
        th = torch.clamp(th, min=-20.0, max=20.0)

        bw = anchors[:, 0].view(1, NUM_ANCHORS, 1, 1) * torch.exp(tw)
        bh = anchors[:, 1].view(1, NUM_ANCHORS, 1, 1) * torch.exp(th)

        x1 = bx - bw / 2
        y1 = by - bh / 2
        x2 = bx + bw / 2
        y2 = by + bh / 2

        class_conf, class_idx = cls_scores.max(dim=-1)
        scores = obj * class_conf

        for b in range(Bc):
            mask = scores[b] > conf_threshold
            if mask.sum() == 0:
                continue
            boxes = torch.stack([x1[b][mask], y1[b][mask], x2[b][mask], y2[b][mask]], dim=1)
            sc = scores[b][mask]
            cl = class_idx[b][mask].float()
            batch_detections[b].append(torch.cat([boxes, sc.unsqueeze(1), cl.unsqueeze(1)], dim=1))

    results = []
    for b in range(B):
        if len(batch_detections[b]) == 0:
            results.append(torch.zeros((0, 6), device=device))
            continue
        dets = torch.cat(batch_detections[b], dim=0)
        keep = batched_nms(dets[:, :4], dets[:, 4], dets[:, 5].long(), nms_threshold)
        keep = keep[:max_detections]
        results.append(dets[keep])
    return results

In [ ]:
!pip install torchmetrics pycocotools --quiet
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchmetrics.classification import BinaryJaccardIndex

class BDD100KEvalDataset(BDD100KDataset):
    def __getitem__(self, idx):
        entry = self.annotations[idx]
        img_path = self.image_path_map[entry['name']]
        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size

        boxes = self._load_boxes(entry)
        box_targets = self._build_targets(boxes, orig_w, orig_h)

        sx, sy = IMG_W / orig_w, IMG_H / orig_h
        if len(boxes) > 0:
            raw = torch.tensor([[b[0]*sx, b[1]*sy, b[2]*sx, b[3]*sy, b[4]] for b in boxes], dtype=torch.float32)
        else:
            raw = torch.zeros((0, 5), dtype=torch.float32)

        image_t = self.transform(image)

        base_name = os.path.splitext(entry['name'])[0]

        possible_mask_filenames = [
            f"{base_name}_{self.split}_id.png",
            f"{base_name}_drivable.png",
            f"{base_name}.png"
        ]

        mask_path = None
        for m_name in possible_mask_filenames:
            candidate_path = os.path.join(self.drivable_mask_dir, m_name)
            if os.path.exists(candidate_path):
                mask_path = candidate_path
                break

        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).resize((IMG_W, IMG_H), Image.Resampling.NEAREST)
            mask = np.array(mask)
            mask = ((mask == 1) | (mask == 2)).astype(np.float32)
            has_mask = True
        else:
            mask = np.zeros((IMG_H, IMG_W), dtype=np.float32)
            has_mask = False

        mask_target = torch.from_numpy(mask).unsqueeze(0)

        return image_t, mask_target, torch.tensor(has_mask, dtype=torch.bool), box_targets, raw

def eval_collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    has_masks = torch.stack([b[2] for b in batch])
    box_targets = [torch.stack([b[3][s] for b in batch]) for s in range(len(GRID_SIZES))]
    raw_boxes = [b[4] for b in batch]
    return images, masks, has_masks, box_targets, raw_boxes

@torch.no_grad()
def evaluate_model(model, dataloader, device, conf_threshold=0.001, nms_threshold=0.5):
    model.eval()
    map_metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
    iou_metric = BinaryJaccardIndex().to(device)

    for images, mask_targets, has_masks, box_targets, raw_boxes_list in dataloader:
        images = images.to(device)
        mask_targets = mask_targets.to(device)
        has_masks = has_masks.to(device)

        seg_pred, yolo_pred = model(images)
        detections = decode_yolo_predictions(yolo_pred, conf_threshold, nms_threshold)

        preds_formatted, targets_formatted = [], []
        for i, det in enumerate(detections):
            preds_formatted.append({'boxes': det[:, :4].cpu(), 'scores': det[:, 4].cpu(), 'labels': det[:, 5].long().cpu()})
            gt = raw_boxes_list[i]
            targets_formatted.append({'boxes': gt[:, :4].cpu(), 'labels': gt[:, 4].long().cpu()})

        map_metric.update(preds_formatted, targets_formatted)

        if has_masks.any():
            valid_seg_preds = seg_pred[has_masks]
            valid_mask_targets = mask_targets[has_masks]

            seg_bin_pred = (torch.sigmoid(valid_seg_preds) > 0.5).int().squeeze(1)
            valid_mask_targets_sq = valid_mask_targets.int().squeeze(1)

            iou_metric.update(seg_bin_pred, valid_mask_targets_sq)

    map_results = map_metric.compute()
    seg_iou = iou_metric.compute()

    print("--- Detection Metrics (mAP) ---")
    print(f"mAP@[.5:.95]: {map_results['map'].item():.4f}")
    print(f"mAP@0.5     : {map_results['map_50'].item():.4f}")
    print(f"mAP@0.75    : {map_results['map_75'].item():.4f}")
    print(f"mAR@100     : {map_results['mar_100'].item():.4f}")
    print("\n--- Segmentation Metrics ---")
    print(f"Drivable-Area IoU: {seg_iou.item():.4f}")

    return map_results, seg_iou

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.amp import GradScaler
import glob
import re
import shutil
from tqdm.auto import tqdm

def plot_training_curves(history, save_dir):
    epochs = range(1, len(history['loss']) + 1)
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history['loss'], label='Total Loss', color='black', linewidth=2)
    plt.plot(epochs, history['unet_loss'], label='Seg (Focal+Dice)', color='blue', linestyle='--')
    plt.plot(epochs, history['yolo_loss'], label='Box (Focal+SmoothL1)', color='red', linestyle='-.')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Panoptic Perception Engine Convergence')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'training_convergence.png'), bbox_inches='tight', dpi=300)
    plt.close()

def get_latest_checkpoint(search_dir):
    if not os.path.exists(search_dir):
        return None

    latest_path = os.path.join(search_dir, 'panoptic_latest.pth')
    if os.path.exists(latest_path):
        return latest_path

    checkpoints = glob.glob(os.path.join(search_dir, 'panoptic_ep*.pth'))
    if not checkpoints:
        return None

    latest_ckpt = max(checkpoints, key=lambda f: int(re.search(r'ep(\d+)', f).group(1)))
    return latest_ckpt

def train_engine(model, dataloader, num_epochs=25, save_dir='./', resume_checkpoint=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = PanopticLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    device_type = device.type
    scaler = GradScaler(device_type, enabled=(device_type == 'cuda'))

    warmup_epochs = 3
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, num_epochs - warmup_epochs), eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

    start_epoch = 0
    history = {'loss': [], 'unet_loss': [], 'yolo_loss': []}

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print(f"Resuming from checkpoint: {resume_checkpoint}")
        # Note: weights_only=False is sometimes required to safely load AdamW states in PyTorch 2.0+
        checkpoint = torch.load(resume_checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        if 'history' in checkpoint:
            history = checkpoint['history']

    print("--- Starting Production Panoptic Training ---")
    for epoch in range(start_epoch, num_epochs):
        model.train()
        epoch_loss, epoch_unet, epoch_yolo = 0.0, 0.0, 0.0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch_idx, (images, mask_targets, has_masks, box_targets) in enumerate(pbar):
            images = images.to(device)
            mask_targets = mask_targets.to(device)
            has_masks = has_masks.to(device)
            box_targets = [b.to(device) for b in box_targets]

            optimizer.zero_grad()
            with torch.autocast(device_type=device_type, enabled=(device_type == 'cuda')):
                unet_preds, yolo_preds = model(images)
                loss, unet_l, yolo_l = criterion(unet_preds, mask_targets, has_masks, yolo_preds, box_targets)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            epoch_unet += unet_l.item()
            epoch_yolo += yolo_l.item()

            pbar.set_postfix(loss=f"{loss.item():.4f}")

        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        avg_loss = epoch_loss / max(1, len(dataloader))
        avg_unet = epoch_unet / max(1, len(dataloader))
        avg_yolo = epoch_yolo / max(1, len(dataloader))

        history['loss'].append(avg_loss)
        history['unet_loss'].append(avg_unet)
        history['yolo_loss'].append(avg_yolo)

        print(f"Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.2e} | Total: {avg_loss:.4f} | Seg: {avg_unet:.4f} | Box: {avg_yolo:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'loss': avg_loss
        }

        temp_save_path = os.path.join(save_dir, 'panoptic_temp.pth')
        latest_save_path = os.path.join(save_dir, 'panoptic_latest.pth')

        torch.save(checkpoint, temp_save_path)
        os.replace(temp_save_path, latest_save_path)

        if (epoch + 1) % 25 == 0:
            torch.save(checkpoint, os.path.join(save_dir, f'panoptic_ep{epoch+1}.pth'))

        plot_training_curves(history, save_dir)

    return history

In [ ]:
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

@torch.no_grad()
def visualize_prediction(model, image_path, device, conf_threshold=0.4, nms_threshold=0.5, save_path=None):
    model.eval()
    orig_image = Image.open(image_path).convert('RGB')

    transform = T.Compose([
        T.Resize((IMG_H, IMG_W)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_tensor = transform(orig_image).unsqueeze(0).to(device)

    seg_pred, yolo_pred = model(input_tensor)
    detections = decode_yolo_predictions(yolo_pred, conf_threshold, nms_threshold)[0].cpu()


    seg_mask = (torch.sigmoid(seg_pred)[0, 0] > 0.5).cpu().numpy()

    display_img = np.array(orig_image.resize((IMG_W, IMG_H)))

    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(display_img)
    ax.imshow(seg_mask, alpha=0.3, cmap='Greens')

    n_warnings = 0
    ignored_classes = [8, 9]

    for det in detections:
        x1, y1, x2, y2, score, cls = det.tolist()
        cls = int(cls)

        cx, cy_bottom = int((x1 + x2) / 2), int(min(y2, IMG_H - 1))

        if cls in ignored_classes:
            is_warning = False
        else:
            is_warning = 0 <= cx < IMG_W and 0 <= cy_bottom < IMG_H and seg_mask[cy_bottom, cx] == 1

        if is_warning:
            n_warnings += 1

        color = 'red' if is_warning else 'lime'
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

        class_name = BDD_CLASSES[cls] if cls < len(BDD_CLASSES) else f"Class {cls}"
        ax.text(x1, max(y1 - 5, 0), f"{class_name} {score:.2f}", color='black', fontsize=9, bbox=dict(facecolor=color, alpha=0.8, pad=1))

    ax.set_title(f"Production Panoptic Output | Detections: {len(detections)} | Hazards: {n_warnings}")
    ax.axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)

    plt.show()
    return detections, seg_mask

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage
import numpy as np
import torchvision.transforms as T
import glob

@torch.no_grad()
def run_testing_pipeline(model, test_dir, output_dir, device, conf_threshold=0.4, nms_threshold=0.5):
    """Runs pure inference on unlabelled test images, tracks objects, and applies the collision hazard system."""
    model.eval()
    os.makedirs(output_dir, exist_ok=True)

    tracker = Collision()

    transform = T.Compose([
        T.Resize((IMG_H, IMG_W)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_images_paths = glob.glob(os.path.join(test_dir, '**', '*.jpg'), recursive=True)[:20]

    if not test_images_paths:
        print(f"Warning: No .jpg images found in {test_dir}. Check your path structure!")
        return

    print(f"Starting testing pipeline on {len(test_images_paths)} images...")

    for img_path in test_images_paths:
        img_name = os.path.basename(img_path)
        orig_image = PILImage.open(img_path).convert('RGB')
        input_tensor = transform(orig_image).unsqueeze(0).to(device)

        seg_pred, yolo_pred = model(input_tensor)
        detections = decode_yolo_predictions(yolo_pred, conf_threshold, nms_threshold)[0].cpu()

        seg_mask_2d = (torch.sigmoid(seg_pred)[0, 0] > 0.5).cpu().numpy()

        boxes = detections[:, :4].tolist() if len(detections) > 0 else []
        tracker.update_tracker(boxes)

        full_detections = detections.tolist() if len(detections) > 0 else []
        mask_4d = torch.tensor(seg_mask_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        warnings = tracker.check_collision(full_detections, mask_4d)

        display_img = np.array(orig_image.resize((IMG_W, IMG_H)))
        fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        ax.imshow(display_img)
        ax.imshow(seg_mask_2d, alpha=0.3, cmap='Greens')

        for det in detections:
            det_list = det.tolist()
            x1, y1, x2, y2, score, cls = det_list

            is_warning = det_list in warnings
            color = 'red' if is_warning else 'lime'

            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

            class_idx = int(cls)
            class_name = BDD_CLASSES[class_idx] if class_idx < len(BDD_CLASSES) else f"Class {class_idx}"
            label = f"{class_name} {score:.2f}"
            ax.text(x1, max(y1 - 5, 0), label, color='black', fontsize=9, bbox=dict(facecolor=color, alpha=0.8, pad=1))

        ax.set_title(f"Test Image: {img_name} | Active Hazards: {len(warnings)}")
        ax.axis('off')

        plt.tight_layout()
        save_file_path = os.path.join(output_dir, f"tested_{img_name}")
        plt.savefig(save_file_path, bbox_inches='tight', dpi=300)
        plt.close()

    print(f"Pipeline complete! Visualizations successfully saved to: {output_dir}")

In [ ]:
import time

@torch.no_grad()
def benchmark_hardware(model, device, input_size=(1, 3, 384, 640), warmups=50, runs=100):
    model.eval()
    dummy_input = torch.randn(input_size, device=device)

    print("\n--- Hardware & Throughput Profiling ---")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total Parameters:        {total_params / 1e6:.2f} M")

    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        _ = model(dummy_input)
        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        print(f"Peak VRAM Footprint:     {peak_vram:.2f} MB")

        for _ in range(warmups):
            _ = model(dummy_input)
        torch.cuda.synchronize()

        start_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
        end_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]

        for i in range(runs):
            start_events[i].record()
            _ = model(dummy_input)
            end_events[i].record()

        torch.cuda.synchronize()
        times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
        avg_latency = np.mean(times)
        fps = 1000.0 / avg_latency
        print(f"Average Latency:         {avg_latency:.2f} ms")
        print(f"99th Percentile Latency: {np.percentile(times, 99):.2f} ms")
        print(f"Engine Throughput:       {fps:.2f} FPS\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Deployment Device: {device}")

model = PanopticPerceptionEngine().to(device)

val_dataset = BDD100KEvalDataset(
    images_dir=IMAGES_VAL,
    det_json_path=DET_JSON_VAL,
    drivable_mask_dir=DRIVABLE_MASK_VAL,
    split='val'
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=eval_collate_fn)

checkpoint_dir = '/content/drive/MyDrive/Panoptic_Checkpoints'
latest_checkpoint_path = get_latest_checkpoint(checkpoint_dir)

if latest_checkpoint_path:
    print(f"Auto-detected previous run. Staging {latest_checkpoint_path} for resumption...")
else:
    print("No previous checkpoints found. Initiating fresh 150-Epoch Production Training Phase...")

train_engine(model, train_loader, num_epochs=150, save_dir=checkpoint_dir, resume_checkpoint=latest_checkpoint_path)

benchmark_hardware(model, device)

print("Initiating Validation Metrics Evaluation...")
evaluate_model(model, val_loader, device)

sample_images = [
    'b1c9c847-3bda4659.jpg',
    'b1c81faa-3df17267.jpg',
    'b1c81faa-c80764c5.jpg',
    'b1cebfb7-284f5117.jpg'
]

output_dir = './predictions'
os.makedirs(output_dir, exist_ok=True)

print("Generating Qualitative Diagnostics...")
for img_name in sample_images:
    img_path = f'{IMAGES_VAL}/{img_name}'
    if os.path.exists(img_path):
        save_loc = os.path.join(output_dir, f'pred_{img_name}')
        visualize_prediction(model, img_path, device, conf_threshold=0.40, save_path=save_loc)


print("\nInitiating Unlabelled Test Pipeline with Collision Tracking...")

TEST_IMAGES_DIR = f'{DATASET_ROOT}/bdd100k/bdd100k/images/100k/test/testA'
TEST_OUTPUT_DIR = './test_predictions'

if os.path.exists(TEST_IMAGES_DIR):
    run_testing_pipeline(
        model=model,
        test_dir=TEST_IMAGES_DIR,
        output_dir=TEST_OUTPUT_DIR,
        device=device,
        conf_threshold=0.40
    )
else:
    print(f"Test directory not found at {TEST_IMAGES_DIR}. Please check your dataset path.")

Deployment Device: cuda
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 172MB/s]


[VAL] Successfully mapped 10000 valid images.
No previous checkpoints found. Initiating fresh 150-Epoch Production Training Phase...
--- Starting Production Panoptic Training ---


Epoch 1/150:   0%|          | 0/2184 [00:00<?, ?it/s]

Epoch [1/150] | LR: 1.00e-05 | Total: 16.7974 | Seg: 0.5421 | Box: 8.1277


...

Epoch 81/150:   0%|          | 0/2184 [00:00<?, ?it/s]

Epoch [81/150] | LR: 4.68e-05 | Total: 3.6301 | Seg: 0.1129 | Box: 1.7586


Epoch 82/150:   0%|          | 0/2184 [00:00<?, ?it/s]

Epoch [82/150] | LR: 4.57e-05 | Total: 3.5957 | Seg: 0.1131 | Box: 1.7413


Epoch 83/150:   0%|          | 0/2184 [00:00<?, ?it/s]